
## Generic AI Agent: watsonx.ai Example


- This notebook demonstrates how to build and deploy a custom AI agent using watsonx.ai
- The agent uses LangGraph and LangChain to create a ReAct agent with custom tools
- This is a template that can be adapted for various use cases

## Install Dependencies and Restart Kernel

In [ ]:
!pip install pydantic==2.11.1 langchain==0.3.22 langchain-core==0.3.51 langchain-ibm==0.3.10 langchain-text-splitters==0.3.8

In [ ]:
# Test for potential package import issues
from langgraph.prebuilt import create_react_agent

## Import Libraries

In [ ]:
import os
from ibm_watsonx_ai import Credentials, APIClient
from ibm_watsonx_ai.deployments import RuntimeContext
import getpass
from dotenv import load_dotenv

## Load Environment Configuration

### Option 1: Load from .env file (if available)

In [ ]:
# Uncomment if using .env file
# load_dotenv('.env')

### Option 2: Enter credentials manually

In [ ]:
# Enter your IBM Cloud API key
CLOUD_API_KEY = getpass.getpass("Enter your IBM Cloud API key: ")
print("API Key received")

In [ ]:
# Configure watsonx.ai parameters
params = {
    "space_id": os.getenv("WATSONX_SPACE_ID", "your-space-id"),  # Replace with your space ID
    "project_id": os.getenv("WATSONX_PROJECT_ID", "your-project-id"),  # Replace with your project ID
    "api_key": CLOUD_API_KEY,
    "service_url": os.getenv("WATSONX_URL", "https://us-south.ml.cloud.ibm.com")  # Update region if needed
}

assert params['api_key'], "API key is required"
print("Parameters configured successfully")

In [ ]:
# Initialize watsonx.ai client
credentials = Credentials(
    url=params['service_url'],
    api_key=params['api_key']
)

space_id = params['space_id']
project_id = params['project_id']
api_client = APIClient(credentials=credentials)

## Define the AI Service

This function will be deployed as a watsonx.ai service. It includes:
- A custom tool that performs data analysis
- A LangGraph ReAct agent that uses the tool
- Request/response handling for deployment

In [ ]:
def gen_ai_service(context, params=params, **custom):
    """
    Generic AI service that can be deployed to watsonx.ai
    
    This example includes:
    - A text analysis tool
    - A LangGraph ReAct agent
    - Streaming and non-streaming response options
    """
    
    from ibm_watsonx_ai import APIClient, Credentials
    from langchain_ibm import WatsonxLLM, ChatWatsonx
    from langchain_core.tools import tool
    from langchain.prompts import PromptTemplate
    from langchain_core.output_parsers import JsonOutputParser
    from langchain_core.messages import AIMessage, HumanMessage
    from pydantic import BaseModel, Field
    from langgraph.prebuilt import create_react_agent
    import json
    
    service_url = params.get("service_url")
    space_id = params.get("space_id")
    project_id = params.get("project_id")
    api_key = params.get("api_key")
    
    api_client = APIClient(
        credentials=Credentials(url=service_url, token=context.generate_token()),
        space_id=space_id
    )
    
    model_id = "meta-llama/llama-3-3-70b-instruct"
    api_client.set.default_space(space_id)
    
    def gen_ai_service_with_langgraph():
        """
        Creates the LangGraph agent with custom tools
        """
        
        def create_llm(
            max_new_tokens=1000,
            min_new_tokens=100,
            temperature=0.7,
            decoding_method='greedy',
            repetition_penalty=1,
            model_id="meta-llama/llama-3-3-70b-instruct"
        ):
            """
            Initialize the LLM with specified parameters
            """
            parameters = {
                "decoding_method": decoding_method,
                "max_new_tokens": max_new_tokens,
                "min_new_tokens": min_new_tokens,
                "repetition_penalty": repetition_penalty,
                "temperature": temperature
            }
            return WatsonxLLM(
                model_id=model_id,
                url=service_url,
                apikey=api_key,
                project_id=project_id,
                params=parameters,
            )
        
        model = create_llm()
        chat = ChatWatsonx(model_id=model_id, watsonx_client=api_client)
        
        # Define Pydantic schema for structured output
        class AnalysisResult(BaseModel):
            """Schema for text analysis results."""
            summary: str = Field(description="Summary of the analysis")
            key_points: list = Field(description="List of key points identified")
            word_count: int = Field(description="Approximate word count")
        
        # Define custom tool
        @tool(parse_docstring=True)
        def analyze_text(text: str) -> dict:
            """
            Analyzes input text and provides structured insights.
            
            This tool performs basic text analysis including:
            - Word count estimation
            - Sentence segmentation
            - Key information extraction
            
            Args:
                text: The text to analyze
            
            Returns:
                A dictionary containing analysis results including word count,
                sentence count, and basic statistics.
            """
            
            print("---------- Analyzing Text ----------")
            print(f"Input text length: {len(text)} characters")
            
            try:
                # Basic text analysis
                words = text.split()
                word_count = len(words)
                
                sentences = text.split('.')
                sentence_count = len([s for s in sentences if s.strip()])
                
                # Calculate average word length
                avg_word_length = sum(len(word) for word in words) / word_count if word_count > 0 else 0
                
                # Create analysis prompt
                analysis_prompt = PromptTemplate(
                    template="""
                    Analyze the following text and provide:
                    1. A brief summary (1-2 sentences)
                    2. 3-5 key points or themes
                    
                    Text: {text}
                    
                    Provide your response in JSON format with keys: 'summary' and 'key_points' (as a list).
                    """,
                    input_variables=["text"]
                )
                
                chain = analysis_prompt | model
                result = chain.invoke({"text": text[:500]})  # Limit text length for analysis
                
                # Try to parse JSON response
                try:
                    parsed = json.loads(result)
                    summary = parsed.get("summary", "Analysis completed")
                    key_points = parsed.get("key_points", [])
                except:
                    # Fallback if JSON parsing fails
                    summary = "Text analysis completed"
                    key_points = ["Statistical analysis performed"]
                
                analysis_result = {
                    "word_count": word_count,
                    "sentence_count": sentence_count,
                    "avg_word_length": round(avg_word_length, 2),
                    "summary": summary,
                    "key_points": key_points,
                    "status": "success"
                }
                
                print("---------- Analysis Complete ----------")
                print(f"Word count: {word_count}")
                print(f"Sentence count: {sentence_count}")
                
                return analysis_result
                
            except Exception as e:
                print(f"Error during analysis: {e}")
                return {
                    "status": "error",
                    "error_message": str(e),
                    "word_count": 0
                }
        
        # Define available tools
        tools = [analyze_text]
        
        # Create LangGraph ReAct agent
        langgraph_agent_executor = create_react_agent(chat, tools)
        
        return langgraph_agent_executor
    
    def convert_messages(messages):
        """
        Convert message format to LangChain message objects
        """
        converted_messages = []
        for message in messages:
            if message["role"] == "user":
                converted_messages.append(HumanMessage(content=message["content"]))
            elif message["role"] == "assistant":
                converted_messages.append(AIMessage(content=message["content"]))
        return converted_messages
    
    def generate(context) -> dict:
        """
        Non-streaming response generation
        """
        api_client.set_token(context.get_token())
        payload = context.get_json()
        messages = payload["messages"]
        
        graph = gen_ai_service_with_langgraph()
        print("Processing request with messages:", convert_messages(messages))
        
        response = graph.invoke({
            "messages": convert_messages(messages)
        })
        
        last_message = response["messages"][-1]
        generated_response = last_message.content
        
        execute_response = {
            "headers": {
                "Content-Type": "application/json"
            },
            "body": {
                "choices": [{
                    "index": 0,
                    "message": {
                        "role": "assistant",
                        "content": generated_response
                    }
                }]
            }
        }
        return execute_response
    
    def generate_stream(context):
        """
        Streaming response generation
        """
        api_client.set_token(context.get_token())
        payload = context.get_json()
        messages = payload["messages"]
        
        graph = gen_ai_service_with_langgraph()
        print("Processing streaming request with messages:", convert_messages(messages))
        
        response = graph.invoke({
            "messages": convert_messages(messages)
        })
        
        last_message = response["messages"][-1]
        generated_response = last_message.content
        
        # Stream response character by character
        for chunk in generated_response:
            chunk_response = {
                "choices": [{
                    "index": 0,
                    "delta": {
                        "role": "assistant",
                        "content": chunk
                    }
                }]
            }
            yield chunk_response
    
    return generate, generate_stream

## Test Locally

Before deploying, test the agent locally to ensure it works correctly

In [ ]:
# Create runtime context for local testing
context = RuntimeContext(api_client=api_client)

# Choose streaming or non-streaming
streaming = False
findex = 1 if streaming else 0

# Get the appropriate function
local_function = gen_ai_service(context, params=params)[findex]

# Test question
question = """Analyze this text: 
Artificial intelligence is transforming how businesses operate. 
Companies are using AI for automation, data analysis, and customer service. 
The technology continues to evolve rapidly, with new applications emerging regularly.
"""

messages = [{"role": "user", "content": question}]
context.request_payload_json = {"messages": messages}

# Execute the function
response = local_function(context)

if streaming:
    print("Streaming response:")
    for chunk in response:
        print(chunk["choices"][0]["delta"]["content"], end="", flush=True)
else:
    print("Response:")
    print(response["body"]["choices"][0]["message"]["content"])

## Deploy to watsonx.ai

The following cells will:
1. Create a software specification with required dependencies
2. Store the AI service
3. Deploy it as an online endpoint

### Configure the YAML Environment File

In [ ]:
api_client.set.default_space(space_id)

In [ ]:
config_yml = """
name: python311
channels:
  - empty
dependencies:
  - pip:
    - langchain-ibm==0.3.5
    - langchain==0.3.22
    - langchain-community==0.3.20
    - langchain-core==0.3.50
    - langgraph==0.2.70
    - pydantic==2.10.6

prefix: /opt/anaconda3/envs/python311
"""

with open("config_generic_agent.yaml", "w", encoding="utf-8") as f:
    f.write(config_yml)

print("Configuration file created: config_generic_agent.yaml")

### Register and Store Environment with Package Extension

In [ ]:
from datetime import datetime

deployment_name = 'generic-agent-deploy-' + datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Deployment name: {deployment_name}")

In [ ]:
# Get base software specification
base_sw_spec_id = api_client.software_specifications.get_id_by_name("runtime-24.1-py3.11")

# Create package extension metadata
meta_prop_pkg_extn = {
    api_client.package_extensions.ConfigurationMetaNames.NAME: deployment_name,
    api_client.package_extensions.ConfigurationMetaNames.DESCRIPTION: "Environment with LangChain and LangGraph",
    api_client.package_extensions.ConfigurationMetaNames.TYPE: "conda_yml"
}

# Store package extension
pkg_extn_details = api_client.package_extensions.store(
    meta_props=meta_prop_pkg_extn,
    file_path="config_generic_agent.yaml"
)
pkg_extn_id = api_client.package_extensions.get_id(pkg_extn_details)

print(f"Package extension ID: {pkg_extn_id}")

### Create Software Specification with Package Extension

In [ ]:
# Create software specification metadata
meta_prop_sw_spec = {
    api_client.software_specifications.ConfigurationMetaNames.NAME: deployment_name,
    api_client.software_specifications.ConfigurationMetaNames.DESCRIPTION: "Software specification for generic AI agent",
    api_client.software_specifications.ConfigurationMetaNames.BASE_SOFTWARE_SPECIFICATION: {"guid": base_sw_spec_id}
}

# Store software specification
sw_spec_details = api_client.software_specifications.store(meta_props=meta_prop_sw_spec)
sw_spec_id = api_client.software_specifications.get_id(sw_spec_details)

# Add package extension to software specification
api_client.software_specifications.add_package_extension(sw_spec_id, pkg_extn_id)

print(f"Software specification ID: {sw_spec_id}")

### Store AI Service

In [ ]:
# Create AI service metadata
meta_props = {
    api_client.repository.AIServiceMetaNames.NAME: f"AI service {deployment_name}",
    api_client.repository.AIServiceMetaNames.SOFTWARE_SPEC_ID: sw_spec_id
}

# Store the AI service
stored_ai_service_details = api_client.repository.store_ai_service(gen_ai_service, meta_props)

print("AI service stored successfully")

### Retrieve AI Service ID

In [ ]:
ai_service_id = api_client.repository.get_ai_service_id(stored_ai_service_details)
print(f"AI service ID: {ai_service_id}")

### Deploy AI Service

In [ ]:
# Create deployment metadata
meta_props = {
    api_client.deployments.ConfigurationMetaNames.NAME: deployment_name,
    api_client.deployments.ConfigurationMetaNames.ONLINE: {},
}

# Create deployment
deployment_details = api_client.deployments.create(ai_service_id, meta_props)

print("Deployment created successfully")

In [ ]:
deployment_id = api_client.deployments.get_id(deployment_details)
print(f"Deployment ID: {deployment_id}")
print(f"\nYour AI agent is now deployed and ready to use!")

## Test Deployed Service

Test the deployed agent by sending requests to the deployment endpoint

In [ ]:
# Example test payload
test_payload = {
    "messages": [
        {
            "role": "user",
            "content": "Analyze this text: Machine learning is a subset of artificial intelligence that focuses on teaching computers to learn from data."
        }
    ]
}

print("Test payload:")
print(test_payload)

In [ ]:
# Send request to deployed service
question = "Analyze this text: Machine learning is a subset of artificial intelligence that focuses on teaching computers to learn from data."
messages = [{"role": "user", "content": question}]
request_payload_json = {"messages": messages}

# Call the deployment
deployments_results = api_client.deployments.run_ai_service(
    deployment_id,
    request_payload_json
)

print("\nDeployment response:")
print(deployments_results)